# 2. Native Deep Agents trial

## tl;dr
Native metadata loading, isolated selection and actual bounded SKILL.md reads are tested. Saved OpenRouter and local Ollama results are reported separately. Model calls require an explicit notebook opt-in; opening/executing this notebook normally spends no API credits.

Fresh setup: follow README.md through `pilot.py vectors` first. Missing live stages remain explicitly unrun. Recorded results are separate under reference_run/.

## Context & Methods
Use the real create_deep_agent / SkillsMiddleware / StateBackend. Main mode offers only select_skills; activation separately permits bounded native read_file.

### Key assumptions
Per-query nested gold-preserving libraries are a conditional distractor study. The same selector, prompts and metadata apply to all methods. Gold remains outside model state.

In [ ]:
import os
from agent_runner import offline_check, Blocked
from trial import smoke, run_trial, activation_smoke
from pilot import ROOT, doctor, rows, read_json
from handoff import report
import pandas as pd
PAID_NOTEBOOK_RUNS = os.getenv("ALLOW_PAID_NOTEBOOK_RUNS") == "1"

## Data
### 1. Check local prerequisites without displaying secrets

In [ ]:
status = doctor()
{k: status[k] for k in ['python', 'env_file_present', 'key_present', 'paid_permission']}

## Results
### 2. Exercise the real native graph with scripted model control

In [ ]:
offline_check()

### 3. Native SKILL.md disclosure, explicitly offline

In [ ]:
activation_smoke(live=False)

### 4. Explicitly opt-in paid stages
A fresh notebook never enables spending. ALLOW_PAID_NOTEBOOK_RUNS=1 is additional to ALLOW_PAID_RUNS=1, a local key and the persistent guard.

In [ ]:
if not PAID_NOTEBOOK_RUNS:
    paid_status = {'status': 'SKIPPED', 'reason': 'Paid notebook opt-in is disabled; live canary/trial not run in this notebook.'}
else:
    try:
        canary = smoke(live=True)
        if not canary.get('canary_correct'):
            raise Blocked('canary_failed')
        comparison = run_trial(quick=True, live=True)
        disclosure = activation_smoke(live=True)
        paid_status = {'status': 'COMPLETED', 'comparison': comparison, 'disclosure': disclosure}
    except Blocked as exc:
        paid_status = {'status': 'BLOCKED', 'reason': str(exc)}
paid_status

### 5. Read report tables from persisted results

In [ ]:
# The original single-model report requires a recorded trial catalogue.
if (ROOT / 'results/trial.jsonl').exists():
    report_summary = report()
    single_model_table = pd.read_csv(ROOT / 'reports/scaling.csv')
else:
    report_summary = {'status': 'NOT RUN', 'reason': 'No single-model trial in this workspace.'}
    single_model_table = pd.DataFrame()
single_model_table if not single_model_table.empty else report_summary

In [ ]:
from IPython.display import Image, display
figure = ROOT / 'reports/figures/candidate_recall.png'
if figure.exists():
    display(Image(filename=str(figure)))
else:
    print('No local trial figure yet. Historical figures are in reference_run/.')

### 6. Local Ollama alternative
By default this section reads saved results only. Set ALLOW_LOCAL_NOTEBOOK_RUNS=1 explicitly to invoke the bounded local runner. It uses installed models and no paid API; completed matching cases are reused.

In [ ]:
from local_runner import run as run_local, report as local_report
if os.getenv("ALLOW_LOCAL_NOTEBOOK_RUNS") == "1":
    local_run_status = run_local()
else:
    local_run_status = {"status": "SKIPPED", "reason": "Local inference opt-in disabled; reading saved evidence only."}
local_run_status

In [ ]:
if rows('results/ollama/canaries.jsonl'):
    local_summary = local_report()
else:
    local_summary = {'selection': [], 'activation': [], 'failures': [],
                     'http_requests': 0, 'api_charge_usd': 0, 'status': 'NOT RUN'}
pd.DataFrame(local_summary['selection'])

In [ ]:
{"activation": local_summary["activation"], "failures": local_summary["failures"], "http_requests": local_summary["http_requests"], "api_charge_usd": local_summary["api_charge_usd"]}

### 7. Five-model OpenRouter follow-up
Same five pilot queries, N=50, listing/BM25/vector/domain-vector. Read saved canary outcomes, failed attempts, exact-match denominators, separate activation and the shared spending ledger. This cell makes no API calls.

In [ ]:
from scripts.report_openrouter import report as paid_report
if any((ROOT / 'results/openrouter' / label / 'trial.jsonl').exists()
       for label in ['qwen38-27b', 'qwen38-flash', 'deepseek41-flash', 'mimo-v2-6-pro', 'glm53-flash']):
    paid_summary = paid_report()
else:
    paid_summary = {'selection': [], 'models': [], 'budget': {},
                    'estimated_cad_at_1_50': None, 'status': 'NOT RUN'}
pd.DataFrame(paid_summary['selection']).reindex(
    columns=['model', 'method', 'attempted', 'valid', 'exact', 'candidate_recall', 'mean_logical_tokens'])

In [ ]:
{"models": paid_summary["models"], "budget": paid_summary["budget"], "estimated_cad": paid_summary["estimated_cad_at_1_50"]}

### 8. Three-query scaling benchmark
Five fixed OpenRouter models at N=50, 250 and 1,000. This bounded expansion reuses matching N=50 evidence and retains invalid outputs and provider exclusions. The following cells regenerate the report from saved data only.

In [ ]:
from scripts.report_scaling import report as scaling_report
if (ROOT / 'manifests/openrouter_scaling.json').exists():
    scaling_summary = scaling_report()
else:
    scaling_summary = {'status': 'NOT RUN', 'summary': [], 'planned': 180,
                       'saved': 0, 'reused_cases': 0, 'attempted': 0, 'valid': 0,
                       'exact': None, 'new_attempts': 0, 'incremental_actual_usd': None,
                       'budget': {}, 'failures': {}}
pd.DataFrame(scaling_summary['summary']).reindex(
    columns=['model', 'N', 'method', 'attempted', 'valid', 'exact', 'candidate_recall'])

In [ ]:
{k: scaling_summary[k] for k in ["planned", "saved", "reused_cases", "attempted", "valid", "exact", "new_attempts", "incremental_actual_usd", "budget", "failures"]}

## Takeaways
A fresh offline run checks native middleware and bounded reads without claiming live accuracy. Live selection and local inference remain unrun until explicitly authorized. Compare any new results with reference_run/ only after checking protocol and denominator differences. No skill execution is measured.